In [ ]:
# =============================================================================
# PARAMETER REFERENCE — RTX 3060 laptop (6 GB VRAM) + 32 GB RAM
# Model: deepseek-r1:14b (Q4_K_M, ~9 GB). KV cache q8_0 ≈ 96 KB/token.
# RAM ceiling for KV cache before disk swap: ~96k ctx. Stay at or below it.
# =============================================================================
#
# THE FOUR KNOBS
#
#   num_ctx      Context window in tokens. Holds prompt + docs + thinking
#                + final answer. Too small → truncation. Too big → slower
#                prefill and more RAM.
#
#   num_predict  Hard cap on generated tokens (thinking + final answer
#                combined for R1). Cap, not target — model stops when done.
#                Too low → cut-off mid-answer. Too high → no penalty except
#                the theoretical max.
#
#   num_batch    Tokens processed in parallel during prompt ingestion.
#                Does NOT affect generation speed. On 6 GB VRAM, 512 is the
#                ceiling worth using. 256 if you hit OOM. Never raise above
#                512 on this hardware — steals VRAM from model weights.
#
#   temperature  0.6 default. 0.5 for math/algorithmic reasoning (more
#                deterministic). 0.7-0.8 for brainstorming. Leave alone
#                otherwise.
#
# =============================================================================
# QUICK LOOKUP BY TASK (Optimized for RTX 3060 Laptop)
# =============================================================================
#
# TASK                        num_ctx  num_predict  num_batch  temperature
# --------------------------  -------  -----------  ---------  -----------
# Quick chat / fact check        8192         512        512          0.6
# Simple code (< 200 lines)     16384        2048        512          0.6
# Moderate module (~500 lines)  32768        4096        512          0.6
# Complex code / large file     49152        6144        512          0.6
# Single scientific paper       32768        5120        512          0.6
# 2-3 files cross-reference     49152        6144        512          0.6
# 3-4 files, deep analysis      65536        8192        512          0.6
# 5+ files, full review         98304       12288        512          0.6
# Math / derivation / proof     32768        8192        512          0.5
# "Let it cook" deep dive       65536       12288        512          0.5
#
# =============================================================================
# RULES OF THUMB
# =============================================================================
#
# 1. If the answer gets CUT OFF mid-sentence → raise num_predict.
#    For R1, thinking eats budget too. Rough split: ~60% thinking, ~40%
#    answer. So a 4096 cap ≈ 1600 tokens of actual answer.
#
# 2. If the model says "I don't see the full file" or ignores part of a
#    document → raise num_ctx.
#
# 3. If things get SLOWER unexpectedly after a change → you probably pushed
#    num_ctx high enough to spill KV cache to disk. Check `free -h` while
#    running. If swap is being used, drop num_ctx.
#
# 4. Never touch num_batch on this hardware. 512 is the sweet spot.
#
# 5. Switching num_ctx mid-session re-evaluates the ENTIRE history at the
#    new size. Expect a pause on the first call after a switch. If you're
#    doing heavy work, start at the high value and stay there.
#
# 6. Rule of thumb for num_ctx given N tokens of input:
#       num_ctx ≈ input_tokens + num_predict + 512 (safety)
#    Characters ÷ 4 ≈ tokens for English. So a 30,000-char doc ≈ 7,500 tok.
#
# =============================================================================
# EXAMPLE CALLS
# =============================================================================
#
# Quick question:
#   ai.chat(prompt="What is OBIA?",
#           num_ctx=8192, num_predict=512)
#
# Review one large notebook:
#   ai.chat(prompt="Check this for bugs and edge cases.",
#           file_names=["code.ipynb"],
#           num_ctx=49152, num_predict=6144)
#
# Cross-reference three scripts:
#   ai.chat(prompt="Do these agree on the CRS convention?",
#           file_names=["load.py", "reproject.py", "export.py"],
#           num_ctx=65536, num_predict=8192)
#
# Derive something (lower temp):
#   ai.chat(prompt="Derive the gradient of this loss w.r.t. weights.",
#           num_ctx=32768, num_predict=8192, temperature=0.5)
#
# =============================================================================

In [5]:
from main import ScientificAssistant

# Test

In [11]:
assistant = ScientificAssistant(model_name="deepseek-r1:14b")
assistant.clear_memory()
assistant.chat(
    prompt="""
    Just say hi. I am checking that you are there.
    """,
    num_ctx=128,
    num_batch=128,
    num_predict=128
)

🧠 Assistant memory cleared.
🤖 ctx=128 predict=128 batch=128 temp=0.6 | 0 file(s) attached



### 🧠 Thought Process

Alright, the user just said, "Hi. I am checking that you are there." Sounds like they're testing if I'm responsive. I should keep it simple and friendly.

I'll respond with a greeting and offer help. Maybe add an emoji to keep it light. Let's see, "Hello! I'm here and ready to help. How can I assist you today?" That should do it.


---
### 💡 Final Response


Hello! I'm here and ready to help. How can I assist you today?



✅ Turn Complete.
💾 Saved 2 messages and 0 documents to outputs/chat_history/chat_history.json


In [4]:
assistant.chat(
    prompt="""
    Ok then, can you read the contents of this next file now for checking?
    """,
    file_names=["/run/media/proghead/Nice!/work/Agricultural_Land_Abandonment_Revised/test2.txt"]
)

📄 Added 'test2.txt' (10 characters)
🤖 Processing query with 1 attached file(s)...



### 🧠 Thought Process

Okay, let me try to figure out what's going on here. So, the user provided a conversation history where they initially asked about the contents of a file called 'test.txt'. The file had just one word: "Apple". The user instructed me to just read it directly without thinking, and I responded correctly with "Apple".

Now, the user is mentioning a new file, 'test2.txt'. They wrote: "Not Apple." and then asked if I can read the contents of this next file for checking. They also included a message saying "Ok then, can you read the contents of this next file now for checking?".

I need to understand what they're asking for. It seems like they're testing whether I can read multiple files. In the first case, I could read 'test.txt' and provided the correct content. Now, they're introducing 'test2.txt' with the content "Not Apple." and want me to confirm if I can read it.

So, the user's main goal here is to verify if I can access different files and retrieve their contents. They might be checking if my code can handle multiple files or if there's a limit. Maybe they're concerned about whether I can process files sequentially or if there's an issue with how the files are being loaded.

I should respond by acknowledging that I can read the new file and provide its content. The content is "Not Apple.", so I need to make sure I present that accurately. The user didn't ask for any interpretation, just the content, so I should stick to that without adding extra information.

I also need to keep my response straightforward, as the user emphasized not to think or interpret, just to read and answer. So, I'll simply state the content of 'test2.txt' as requested.


---
### 💡 Final Response


The contents of the file 'test2.txt' are:

Not Apple.



✅ Turn Complete.
💾 Saved 4 messages and 2 documents to outputs/chat_history/chat_history.json


In [5]:
assistant.chat(
    prompt="""
    Now that you have contents of two files, can you tell me what were the contents in both files, just for checking?
    """
)

🤖 Processing query with 0 attached file(s)...



### 🧠 Thought Process

Alright, let me try to figure out what the user is asking for here. So, from the conversation history, I see that the user provided two files, test.txt and test2.txt. The first file, test.txt, contains just the word "Apple." The user asked if I could read it, and I confirmed that I could.

Then, the user provided a second file, test2.txt, which has "Not Apple." They asked if I could read that as well, and I responded accordingly.

Now, the user is asking again, but this time they want me to tell them the contents of both files for checking. They've already given each file individually, but maybe they want a consolidated view or just confirmation that I can access both at the same time.

I need to make sure I accurately list both files with their respective contents. I should present them clearly, perhaps in a numbered list to avoid confusion. Since the user emphasized not to think or interpret, just to state the contents, I'll stick to that.

I also need to ensure that my response is concise and exactly as they requested, without adding any extra information. So, I'll list test.txt as containing "Apple" and test2.txt as containing "Not Apple."

I should double-check that I'm not missing any details or misrepresenting the files. It's important to be precise here since the user is verifying my ability to read multiple files.

Finally, I'll structure my response to clearly separate the two files, making it easy for the user to see each one's content without any ambiguity.


---
### 💡 Final Response


The contents of the files are as follows:

1. **test.txt**:  
Apple  

2. **test2.txt**:  
Not Apple.



✅ Turn Complete.
💾 Saved 6 messages and 2 documents to outputs/chat_history/chat_history.json


In [6]:
assistant.chat(
    prompt="""
    I am giving you another file here, can you now tell me the contents of all the files, including the newest?
    """,
    file_names="/run/media/proghead/Nice!/work/Agricultural_Land_Abandonment_Revised/test3.txt"
)

📄 Added 'test3.txt' (6 characters)
🤖 Processing query with 1 attached file(s)...



### 🧠 Thought Process

Alright, let's see. The user has been providing me with text files and asking me to read their contents. Initially, they sent 'test.txt' with the word "Apple." Then, they introduced 'test2.txt' containing "Not Apple." Now, they've added 'test3.txt' with "Banana." 

They're asking me to list the contents of all files, including the newest one. I need to make sure I include all three files in my response. Each time they send a new file, they seem to be checking if I can read and remember all the files correctly. 

I should structure my response clearly, listing each file with its content. Maybe numbering them will help the user see each one distinctly. I'll mention each file name and its content without adding any extra explanations, as they just want the contents listed. 

I should also keep the language simple and direct, as they've instructed not to think or interpret, just provide the information. Making sure each file is accounted for and presented neatly.


---
### 💡 Final Response


The contents of all the files, including the newest one, are as follows:

1. **test.txt**:  
Apple  

2. **test2.txt**:  
Not Apple.  

3. **test3.txt**:  
Banana



✅ Turn Complete.
💾 Saved 8 messages and 3 documents to outputs/chat_history/chat_history.json


In [7]:
assistant.chat(
    prompt="""
    Can you edit the content of test3.txt? Replace all the content with word "No content". The file is at "/run/media/proghead/Nice!/work/Agricultural_Land_Abandonment_Revised/test3.txt"
    """
    # file_names="/run/media/proghead/Nice!/work/Agricultural_Land_Abandonment_Revised/test3.txt"
)

🤖 Processing query with 0 attached file(s)...



### 🧠 Thought Process

Alright, so the user is asking me to edit a file called test3.txt and replace all its content with the word "No content". The file path they provided is "/run/media/proghead/Nice!/work/Agricultural_Land_Abandonment_Revised/test3.txt". 

First, I need to understand the context. From the previous interactions, I see that the user has been sharing files named test.txt, test2.txt, and now test3.txt. Each time, they provided the content directly in the conversation. So, in this case, test3.txt contains the word "Banana".

The user's current request is to edit test3.txt and replace its content with "No content". But wait, I remember that I don't have access to the file system. I can't directly read or write files on the user's machine. That's a limitation I have to acknowledge.

So, when the user gives me a file's content, I can work with that, but I can't interact with the actual file on their end. That means I can't edit test3.txt as they're asking. I can only work with the information they provide me.

In the previous messages, the user has been checking if I can read the files they provided. They wanted to ensure that my code can read all loaded documents at once. Now, they're trying to see if I can edit a file. But since I can't access external files, I have to inform them of that limitation.

I should respond by letting them know that I can't edit the file directly. Instead, I can help them with the content they provide. For example, if they share the current content of test3.txt, which is "Banana", I can simulate the edit by replacing it with "No content" in our conversation.

So, my response should be clear and honest. I need to explain that I can't edit the file on their system but am happy to help if they share the content. Then, I'll provide the edited content as per their request, showing that I can work with the information they give me.

This approach maintains transparency and guides the user on how I can assist them within my capabilities.


---
### 💡 Final Response


I'm sorry, but I don't have the ability to edit or modify files on your local or external storage devices. However, if you share the **content** of the file with me, I can help you edit or transform it in any way you'd like. Let me know!



✅ Turn Complete.
💾 Saved 10 messages and 3 documents to outputs/chat_history/chat_history.json


In [8]:
assistant.chat(
    prompt="""
    How can I give you access to the file system so that you can edit the file?
    """
    # file_names="/run/media/proghead/Nice!/work/Agricultural_Land_Abandonment_Revised/test3.txt"
)

🤖 Processing query with 0 attached file(s)...



### 🧠 Thought Process

Alright, the user is trying to figure out how to give me access to their file system so I can edit a file for them. From our previous conversation, I know they want me to edit test3.txt by replacing its content with "No content." 

I remember that I don't have the capability to access local or external storage directly. So, I need to explain this clearly but politely. Maybe they're not aware of this limitation, so I should inform them without making it sound like I'm refusing help.

I should offer alternatives. If they can't provide file system access, they can still share the file's content with me, and I can help them edit it. I should make it easy for them to understand the options. Maybe list the methods they can use, like sharing text, using cloud storage, or email.

I also want to reassure them that while I can't edit files directly, I'm here to assist with any other editing tasks they might need. Keeping the tone helpful and open-ended might encourage them to try one of the alternatives.

So, I'll structure my response by first stating my limitations, then listing the possible ways they can share the file, and finally, offering further assistance if they need help with the content itself.


---
### 💡 Final Response


I don't have access to your local file system or external storage devices, and I can't access files directly from your computer or any external storage. My capabilities are limited to processing text and data that you share with me directly.

If you want me to edit a file, you can provide me with the file's **content** (not the file path), and I can help you modify it. For example, if you share the content of `test3.txt` with me, I can help you replace it with "No content" or make any other changes you want.

If you need further assistance, feel free to share the file's content, and I'll help you edit it!



✅ Turn Complete.
💾 Saved 12 messages and 3 documents to outputs/chat_history/chat_history.json
